In [14]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
#%matplotlib widget
%matplotlib inline
from figures import *
import pandas as pd
import json
import numpy as np
from sklearn.metrics import (
	roc_auc_score,
	brier_score_loss,
	confusion_matrix,
	f1_score,
	precision_score,
	recall_score,
	accuracy_score
)

history_cols = ["Timestamp","Model","OuterFold","HPset","Epoch","TrainLoss","TrainAcc","ValLoss","ValAcc",
		   "AUC", "Brier","EMA_ValLoss","LR","NoImprove","LrDrop","EsTriggered","BestValLoss","BestEpoch"]


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:

history_cols = ["Timestamp","Model","OuterFold","HPset","Epoch","TrainLoss","TrainAcc","ValLoss","ValAcc",
		   "AUC", "Brier","EMA_ValLoss","LR","NoImprove","LrDrop","EsTriggered","BestValLoss","BestEpoch"]

history = pd.read_csv("OUTER_epochs.csv", engine='python')
print(history.columns)


summary = pd.read_csv("OUTER_summary.csv", sep=r'\s*,\s*', engine='python')
print(summary.columns)

evaluate = pd.read_csv("OUTER_evaluate.csv", engine='python')
print(evaluate.columns)


#summary_mean_std = pd.read_csv("OUTER_summary_mean_std.csv", sep=r'\s*,\s*', engine='python')
#print(summary_mean_std.columns)


Index(['Timestamp', 'Model', 'OuterFold', 'HP', 'Epoch', 'TrainLoss',
       'TrainAcc', 'ValLoss', 'ValAcc', 'AUC', 'Brier', 'EMA_ValLoss', 'LR',
       'NoImprove', 'LrDrop', 'EsTriggered', 'BestValLoss', 'BestEpoch'],
      dtype='object')
Index(['Timestamp', 'model_name', 'out', 'HP', 'cid', 'y_true', 'y_hat',
       'y_prob', 'y_logit', 'TH', 'LR', 'WD', 'DR'],
      dtype='object')


In [ ]:
eval = pd.read_csv("OUTER_evaluat.csv", sep=r'\s*;\s*', engine='python')
print(eval.columns)
eval.to_csv('OUTER_evaluate.csv', index=False)
evaluate = pd.read_csv("OUTER_evaluate.csv", engine='python')
print(evaluate.columns)


Index(['Timestamp', 'model_name', 'out', 'HP', 'cid', 'y_true', 'y_hat',
       'y_prob', 'y_logit', 'TH', 'LR', 'WD', 'DR'],
      dtype='object')
Index(['Timestamp', 'model_name', 'out', 'HP', 'cid', 'y_true', 'y_hat',
       'y_prob', 'y_logit', 'TH', 'LR', 'WD', 'DR'],
      dtype='object')


In [ ]:
#summary = pd.read_csv("OUTER_summary.csv", sep=r'\s*,\s*', engine='python')
#print(summary.columns)

with open("OUTER_summary.jsonl", 'r') as jf:
	summary = [json.loads(line) for line in jf]
summary = pd.DataFrame(summary)
summary.to_csv('OUTER_summary.csv', index=False)


In [19]:

# --- (Use the loading code from Step 1) ---

with open("OUTER_evaluation.jsonl", 'r') as f:
	data_list = [json.loads(line) for line in f]
df = pd.DataFrame(data_list)

# Define a function to calculate all metrics for one row of the DataFrame
def calculate_metrics(row):
	y_true = np.array(row['y_true'])
	y_prob = np.array(row['y_prob'])
	y_hat = np.array(row['y_hat'])

	# Handle cases with only one class present in y_true
	if len(np.unique(y_true)) < 2:
		return {
			'auc': np.nan, 'brier_score': np.nan, 'f1': np.nan,
			'accuracy': np.nan, 'precision': np.nan, 'recall': np.nan,
			'TN': np.nan, 'FP': np.nan, 'FN': np.nan, 'TP': np.nan
		}

	# Calculate metrics
	auc = roc_auc_score(y_true, y_prob)
	brier = brier_score_loss(y_true, y_prob)
	f1 = f1_score(y_true, y_hat)
	accuracy = accuracy_score(y_true, y_hat)
	precision = precision_score(y_true, y_hat, zero_division=0)
	recall = recall_score(y_true, y_hat, zero_division=0)

	# Confusion matrix components
	# .ravel() flattens the 2x2 matrix into a 1D array: [TN, FP, FN, TP]
	tn, fp, fn, tp = confusion_matrix(y_true, y_hat).ravel()

	# Return a dictionary of the calculated metrics
	return {
		'auc': auc,
		'brier_score': brier,
		'f1': f1,
		'accuracy': accuracy,
		'precision': precision,
		'recall': recall,
		'TN': int(tn), # Convert from numpy.int64 to standard int
		'FP': int(fp),
		'FN': int(fn),
		'TP': int(tp)
	}

# Apply the function to each row of the DataFrame
# The result is a Series of dictionaries
metrics_series = df.apply(calculate_metrics, axis=1)

# Convert the Series of dictionaries into a new DataFrame
metrics_df = pd.json_normalize(metrics_series)

# Combine the new metrics with your original DataFrame
results_df = pd.concat([df, metrics_df], axis=1)

# You can drop the list-like columns for a cleaner summary view
final_summary = results_df.drop(columns=['case_ids', 'y_true', 'y_prob', 'y_logit', 'y_hat'])

print("\n--- Final Summary with Calculated Metrics ---")
#print(final_summary) summary.to_csv('OUTER_summary.csv', index=False)
final_summary.to_csv('OUTER_results.csv', index=False)



--- Final Summary with Calculated Metrics ---
